In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Milestone 4.3: Bachelier Comparison

This notebook compares Black, shifted Black, and Bachelier pricing across rate regimes. The objective is to understand why a normal-vol framework becomes important when rates are low or negative.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from swaption_pricing.bachelier import bachelier_option_value, price_swaption_bachelier
from swaption_pricing.black76 import price_shifted_black, price_swaption, price_swaption_shifted_black
from swaption_pricing.curve_bootstrap import bootstrap_zero_curve
from swaption_pricing.types import MarketQuote, SwaptionSpec

## 1. Positive-Rate Benchmark

In [ ]:
quotes = [
    MarketQuote(instrument_type='deposit', maturity=1.0, rate=0.0420),
    MarketQuote(instrument_type='swap', maturity=2.0, rate=0.0415, pay_frequency=1),
    MarketQuote(instrument_type='swap', maturity=3.0, rate=0.0410, pay_frequency=1),
    MarketQuote(instrument_type='swap', maturity=4.0, rate=0.0408, pay_frequency=1),
    MarketQuote(instrument_type='swap', maturity=5.0, rate=0.0405, pay_frequency=1),
    MarketQuote(instrument_type='swap', maturity=6.0, rate=0.0403, pay_frequency=1),
    MarketQuote(instrument_type='swap', maturity=7.0, rate=0.0402, pay_frequency=1),
]
curve = bootstrap_zero_curve(quotes)
black_vol = 0.20
normal_vol = 0.01
shift = 0.03

positive_rows = []
for option_type in ['payer', 'receiver']:
    spec = SwaptionSpec(notional=10_000_000.0, expiry=2.0, tenor=5.0, strike=0.0400, pay_frequency=1, option_type=option_type)
    positive_rows.append({
        'option_type': option_type,
        'black_price': price_swaption(curve, spec, black_vol),
        'shifted_black_price': price_swaption_shifted_black(curve, spec, black_vol, shift),
        'bachelier_price': price_swaption_bachelier(curve, spec, normal_vol),
    })

positive_df = pd.DataFrame(positive_rows)
positive_df

## 2. Negative-Rate Direct Payoff Comparison

When the forward and strike are negative, standard lognormal Black is not directly available. We compare shifted Black and Bachelier at the forward-rate payoff layer.

In [ ]:
negative_forward = -0.0020
negative_strike = -0.0010
negative_expiry = 2.0
negative_shift = 0.03

negative_rows = []
for option_type in ['payer', 'receiver']:
    negative_rows.append({
        'option_type': option_type,
        'shifted_black_payoff': price_shifted_black(negative_forward, negative_strike, negative_expiry, black_vol, negative_shift, option_type),
        'bachelier_payoff': bachelier_option_value(negative_forward, negative_strike, negative_expiry, normal_vol, option_type),
    })

negative_df = pd.DataFrame(negative_rows)
negative_df

In [ ]:
ax = positive_df.plot(x='option_type', y=['black_price', 'shifted_black_price', 'bachelier_price'], kind='bar', figsize=(8, 4), title='Positive-Rate Model Comparison')
ax.set_ylabel('Price')
plt.xticks(rotation=0)
plt.show()

ax = negative_df.plot(x='option_type', y=['shifted_black_payoff', 'bachelier_payoff'], kind='bar', figsize=(8, 4), title='Negative-Rate Payoff Comparison')
ax.set_ylabel('Payoff Layer Value')
plt.xticks(rotation=0)
plt.show()

## 3. Interpretation

- Black is natural when rates are comfortably positive.
- Shifted Black extends the lognormal framework into low-rate and negative-rate settings by moving the state variable.
- Bachelier avoids the positivity constraint altogether because it works with normal moves in the forward rate.
- In low-rate regimes, model choice becomes a market-convention and risk-interpretation question, not just a numerical implementation detail.